<a href="https://colab.research.google.com/github/TonyQ2k3/pytorch-training/blob/main/notebooks/stock_forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mlflow

In [ ]:
!pip show pyspark

In [ ]:
import mlflow
import mlflow.pyfunc
import mlflow.spark
import os

from pyspark.sql import SparkSession, Row
from pyspark.ml import PipelineModel

In [ ]:
import pandas as pd

data = pd.read_csv('/content/Google Pixel_2025-05-15_00-16-54.csv')
data.rename(columns={'text': 'Text'}, inplace=True)
data['Label'] = 1.0

In [ ]:
data.head()

In [ ]:
def load_model_from_mlflow(model_uri):
    """
    Load a model from MLflow.
    :param model_uri: URI of the model in MLflow.
    :return: Loaded model.
    """
    # Load the model
    model = mlflow.spark.load_model(model_uri)
    return model

In [ ]:
mlflow.set_tracking_uri("https://dagshub.com/TranChucThien/kltn-sentiment-monitoring-mlops.mlflow")
spark = SparkSession.builder \
.appName("Load CountVectorizer_Model from MLflow") \
.getOrCreate()

# Load the model from MLflow
model_uri = "models:/CountVectorizer_Model/83"
model = load_model_from_mlflow(model_uri)

In [ ]:
# Tạo DataFrame
df = spark.createDataFrame(data)

# Chạy transform qua model đã load
predictions = model.transform(df)

# Hiển thị kết quả
# predictions.show(truncate=False)
predictions.select("Text", "prediction").show(truncate=False)

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.functions import col

In [ ]:
def map_class(value):
  class_index_mapping = { 0: "Negative", 1: "Positive", 2: "Neutral" }
  return class_index_mapping[int(value)]

In [ ]:
predictions = predictions.withColumn("class", udf(map_class)(col("prediction")))

In [ ]:
predictions.select("class", "Text").show(truncate=False)